# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# Load data
csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))
if not csv_files:
    import subprocess, os
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/M-Raqib-Usman/ML_Internship.git", "repo_temp"], check=True)
    os.chdir("repo_temp")
    csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))

df = pd.read_csv(csv_files[0])
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("Loaded:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Loaded: (30000, 45)
Declining rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest (classification → ranking by probability)**

My lane is ranking pages for refresh review ("which ones first?").  
A Random Forest gives a probability score I can rank by, handles mixed numeric signals, and is stronger than a single tree while still allowing feature-importance interpretation.  
I will compare it against the hand-written baseline from Week 4 on the same split and the same metric (Precision@50).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-holdout split (GroupShuffleSplit on client_id)**

Pages from the same client share the same site, content strategy, and seasonality.  
A normal random split would let the model memorize client patterns and look better than it really is.  
Holding out entire clients tests whether the model generalizes to new sites — the real use case.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features (safe – no trend_pct / trend_direction)
feature_cols = [
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count", "engagement_rate", "sessions_90d"
]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
groups = df["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}")
print(f"Train clients: {df.iloc[train_idx]['client_id'].nunique()} | Test clients: {df.iloc[test_idx]['client_id'].nunique()}")
print(f"Test declining rate: {y_test.mean():.3f}")

Train rows: 23,837 | Test rows: 6,163
Train clients: 25 | Test clients: 7
Test declining rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline (same rule as Week 4) ---
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

baseline_test_scores = df.iloc[test_idx]["baseline_score"].values

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Comparison table ---
base_rate = y_test.mean()
results = pd.DataFrame({
    "Method": ["Base rate (random)", "Hand-rule baseline", "Random Forest"],
    "Precision@20": [
        base_rate,
        precision_at_k(baseline_test_scores, y_test, 20),
        precision_at_k(rf_scores, y_test, 20)
    ],
    "Precision@50": [
        base_rate,
        precision_at_k(baseline_test_scores, y_test, 50),
        precision_at_k(rf_scores, y_test, 50)
    ],
    "ROC-AUC": [
        0.5,
        roc_auc_score(y_test, baseline_test_scores) if baseline_test_scores.std() > 0 else 0.5,
        roc_auc_score(y_test, rf_scores)
    ]
}).round(3)

print(results.to_string(index=False))
print("\nHigher Precision@50 = better ranking for the editor's limited time.")

            Method  Precision@20  Precision@50  ROC-AUC
Base rate (random)         0.511         0.511    0.500
Hand-rule baseline         0.800         0.700    0.500
     Random Forest         0.650         0.640    0.596

Higher Precision@50 = better ranking for the editor's limited time.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("=== Top features the model leans on ===")
print(imp.round(3))

# Error look
test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf_scores
test_df["error"] = (test_df["rf_score"] > 0.5).astype(int) != test_df["is_declining_label"]

print("\n=== Where the model is most wrong ===")
print("False positives (predicted decline but not declining) – high score but label=0:")
print(test_df[(test_df["rf_score"] > 0.7) & (test_df["is_declining_label"] == 0)]
      [["impressions_90d", "days_since_last_update", "avg_position", "ctr", "rf_score"]].head(5))

print("\nFalse negatives (missed real declines) – low score but label=1:")
print(test_df[(test_df["rf_score"] < 0.3) & (test_df["is_declining_label"] == 1)]
      [["impressions_90d", "days_since_last_update", "avg_position", "ctr", "rf_score"]].head(5))

=== Top features the model leans on ===
impressions_90d           0.289
avg_position              0.193
content_age_days          0.178
word_count                0.129
ctr                       0.069
days_since_last_update    0.060
sessions_90d              0.060
engagement_rate           0.023
dtype: float64

=== Where the model is most wrong ===
False positives (predicted decline but not declining) – high score but label=0:
     impressions_90d  days_since_last_update  avg_position   ctr  rf_score
13               307                     103          39.8  0.00  0.769813
26              2426                      13          30.0  0.12  0.700761
78                59                      92           8.7  0.00  0.761116
356            11795                     104          29.9  0.15  0.792029
487             6089                      25          12.8  0.10  0.706757

False negatives (missed real declines) – low score but label=1:
     impressions_90d  days_since_last_update  avg_posit

The model leans most on impressions volume, position, and freshness — which matches the domain.  
False positives often look like high-volume pages that are still stable; false negatives are often low-volume pages where the trend signal is noisy.  
This is expected: the ranking is decision-support, not a perfect predictor.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.